In [ ]:
import requests
import os
import re
import gzip
import shutil
import pandas as pd
from collections import defaultdict

In [ ]:
BASE_URL = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/"
# rename output directory to whatever you want
outdir = "stormevents"
os.makedirs(outdir, exist_ok=True)

In [ ]:
# this gets details, fatalities, and locations files. details is probably the main thing you want, but might as well grab all.
html = requests.get(BASE_URL).text

pattern = r"StormEvents_(details|fatalities|locations)-ftp_v1.0_d(\d{4})_c(\d{8})\.csv\.gz"
matches = re.findall(pattern, html)

latest_files = defaultdict(dict)

for ftype, year, catdate in matches:
    key = (year, ftype)
    filename = f"StormEvents_{ftype}-ftp_v1.0_d{year}_c{catdate}.csv.gz"
    current = latest_files[year].get(ftype)

    if not current or catdate > re.search(r"c(\d+)", current).group(1):
        latest_files[year][ftype] = filename

for year, files in latest_files.items():
    for ftype, filename in files.items():
        url = BASE_URL + filename
        path = os.path.join(outdir, filename)

        print(f"Downloading {filename}")
        try:
            r = requests.get(url, stream=True)
            if r.ok:
                with open(path, "wb") as f:
                    for chunk in r.iter_content(8192):
                        f.write(chunk)
            else:
                print(f"Failed: {filename} ({r.status_code})")
        except Exception as e:
            print(f"Error downloading {filename}: {e}")


In [ ]:
dir_name = outdir

from pathlib import Path

# unzip archives and replace them with the extracted csvs
def gz_extract(directory):
    # adjusted from https://gist.github.com/kstreepy/a9800804c21367d5a8bde692318a18f5

    for gz_file in Path(directory).glob("*.gz"):
        csv_file = gz_file.with_suffix("")  # removes .gz
        if csv_file.exists():
            continue

        with gzip.open(gz_file, "rb") as f_in, open(csv_file, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        gz_file.unlink()

        
gz_extract(dir_name)

In [ ]:
# convert the files to dataframes, one df per type of file
typed_dfs = defaultdict(list)
for filename in os.listdir(outdir):
    filepath = os.path.join(outdir, filename)
    if not filename.endswith(".csv"):
        continue

    if "details" in filename:
        ftype = "details"
    elif "fatalities" in filename:
        ftype = "fatalities"
    elif "locations" in filename:
        ftype = "locations"
    else:
        continue

    df = pd.read_csv(filepath, low_memory=False)
    typed_dfs[ftype].append(df)

In [ ]:
# combine all the years
details_df = pd.concat(typed_dfs["details"], ignore_index=True)
fatalities_df = pd.concat(typed_dfs["fatalities"], ignore_index=True)
locations_df = pd.concat(typed_dfs["locations"], ignore_index=True)

In [ ]:
# filter it down to tornadoes only
tornado_details_df = details_df[details_df.EVENT_TYPE == "Tornado"]
tornado_details_df

In [ ]:
tornado_details_df.DAMAGE_PROPERTY.value_counts()

In [ ]:
# need this to get the other 2 df filtered too
tornado_event_ids = tornado_details_df["EVENT_ID"].unique()

In [ ]:
tornado_fatalities_df = tornado_fatalities = fatalities_df[fatalities_df["EVENT_ID"].isin(tornado_event_ids)]
tornado_fatalities_df

In [ ]:
tornado_locations_df = locations_df[locations_df["EVENT_ID"].isin(tornado_event_ids)]
tornado_locations_df

In [ ]:
# crops were making the file saving mess up, so instead of '5K' it's 5000, etc.
def parse_damage(val):
    if pd.isna(val) or not str(val).strip():
        return None

    val = str(val).strip().upper()

    try:
        # Handle 'K', 'M', 'B' with and without decimals
        if val.endswith("K") and len(val) > 1:
            return float(val[:-1]) * 1_000
        elif val.endswith("M") and len(val) > 1:
            return float(val[:-1]) * 1_000_000
        elif val.endswith("B") and len(val) > 1:
            return float(val[:-1]) * 1_000_000_000
        else:
            return float(val)
    except ValueError:
        return None

tornado_details_df["DAMAGE_CROPS"] = tornado_details_df["DAMAGE_CROPS"].apply(parse_damage)
tornado_details_df["DAMAGE_PROPERTY"] = tornado_details_df["DAMAGE_PROPERTY"].apply(parse_damage)

In [ ]:
# save to parquet files
tornado_details_df.to_parquet('processed_files/tornado_details.parquet')
tornado_fatalities_df.to_parquet('processed_files/tornado_fatalities.parquet')
tornado_locations_df.to_parquet('processed_files/tornado_locations.parquet')

In [ ]:
tornado_details_df_cleaned = tornado_details_df.dropna(axis=1, how='all')
tornado_details_df_cleaned

In [ ]:
tornado_details_df.columns[tornado_details_df.isna().all()].tolist()

In [ ]:
import nbformat

def repair_notebook(path_in, path_out=None):
    nb = nbformat.read(path_in, as_version=4)

    for i, cell in enumerate(nb.cells):
        cell.setdefault("metadata", {})

        if cell.cell_type == "code":
            cell.setdefault("execution_count", None)
            cell.setdefault("outputs", [])
            cell.setdefault("source", "")

        elif cell.cell_type == "markdown":
            cell.setdefault("source", "")

    nb.metadata.setdefault("language_info", {})
    nb.metadata.setdefault("kernelspec", {})
    
    path_out = path_out or path_in
    nbformat.write(nb, path_out)

repair_notebook("storm_events.ipynb")